# Create model for prediction of gamma shower

## Load modules and data

In [ ]:
# import modules
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline 

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, validation_curve, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, make_scorer
from sklearn.base import clone

import optuna 
from optuna.samplers import TPESampler
import time

from src.features import engineer_features

In [35]:
# ignore Data Conversion Warnings
from sklearn.exceptions import DataConversionWarning
import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)

In [36]:
# show three digits
pd.set_option('display.float_format', '{:,.3f}'.format)
pd.set_option('display.precision', 3)
np.set_printoptions(precision=3)

In [37]:
# set column names
columns = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
    "class"
]

# load data
df = pd.read_csv(
    r"../data/raw/magic04.data",
    header=None,
    names=columns
)

# remove dublicates before train-test-split
df = df.drop_duplicates().reset_index(drop=True)

# convert target from g/h --> 1/0
df["class"] = df["class"].map({
    "g": 1,   # gamma = signal
    "h": 0    # hadron = background
})

features = df.iloc[:, :-1]
target = df.iloc[:, -1]

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.1, random_state=42)

In [38]:
display(target_train)

15421    0
6829     1
11808    1
7765     1
12077    1
        ..
11284    1
11964    1
5390     1
860      1
15795    0
Name: class, Length: 17014, dtype: int64

In [39]:
# save features_test as 'features_test.csv'
features_train.to_csv('../data/raw/features_train_unprocessed.csv', index=False)

In [40]:
# save target_train as 'target_train.csv'
target_train.to_csv('../data/raw/target_train_unprocessed.csv', index=False)

In [41]:
# save features_test as 'features_test.csv'
features_test.to_csv('../data/rawfeatures_test_unprocessed.csv', index=False)

In [42]:
# save target_test as 'target_test.csv'
target_test.to_csv('../data/raw/target_test_unprocessed.csv', index=False)

## Scoring
Bei dem gegebenen Datensatz ist die accuracy als Gütemaß nicht aussagekräftig. Eine Klassifizierung eines Backround-Events als Gamma-Event ist hier viel schwerwiegender als eine fälschliche Klassifizierung eines Gamma-Events als Backround-Events.
Hier kommt es besonders darauf an, für verschiedene Grenzwerte 0.01, 0.02, 0.05, 0.1 und 0.2 (je nach Experiment) für die Wahrscheinlichkeit ein Backround-Event als Signal zu klassifizieren einen möglichst hohen Wert bei der ROC-Curve zu erhalten. Sieht man ein Gamma-Ereignis als Klasse  1 und ein Backround-Event als Klasse 0 an, so ergibt FPR (False Positiv Rate) den Anteil der Hadron-Ereignisse an, die fälschlicherweise als Gamma klassifiziert werden und TRP (True Positiv Rate / Recall) den Anteil der Gamma-Ereignisse, die korrekt als Gama identifiziert werden. Für die jeweiligen Grenzwerte der FPR, soll also die TPR (also die gamma-Effiezienz) möglichst groß werden. 
Für dieses Projekt wird deshalb ein eigenes Scoring definiert.
Als primäre Auswahlmetrik wird hier 5% verwendet, die anderen Metriken werden jedoch zusätzlich betrachtet

In [43]:
# true positiv rate at a given false positive rate, 0.05 as default
def tpr_at_fpr(target, target_pred, max_fpr=0.05):
    fpr, tpr, _ = roc_curve(target, target_pred)

    return np.max(tpr[fpr <= max_fpr])

In [44]:
# define scorer for different thresholds of fpr
scoring = {
    "roc_auc": "roc_auc",

    "tpr_fpr_001": make_scorer(
        tpr_at_fpr,
        response_method=("decision_function", "predict_proba"),   # use continouus model-score, if decision_function not supportet, use predict_proba
        max_fpr=0.01
    ),

    "tpr_fpr_002": make_scorer(
        tpr_at_fpr,
        response_method=("decision_function", "predict_proba"),
        max_fpr=0.02
    ),

    "tpr_fpr_005": make_scorer(
        tpr_at_fpr,
        response_method=("decision_function", "predict_proba"),
        max_fpr=0.05
    ),

    "tpr_fpr_010": make_scorer(
        tpr_at_fpr,
        response_method=("decision_function", "predict_proba"),
        max_fpr=0.10
    ),

    "tpr_fpr_020": make_scorer(
        tpr_at_fpr,
        response_method=("decision_function", "predict_proba"),
        max_fpr=0.20
    )
}


In diesem Notebook wird zunächst nur der scorer "tpr_fpr_020" verwendet

## Data Preparation

### Datatype transformation

Es ist keine weitere Umwandlung der Trainingsdaten notwendig (Umwandlung Target g/h --> 1/0 siehe oben)

### Data Imputation

Der Datensatz beinhaltet keine fehlenden Werte, es müssen also keine Daten imputiert werden

### Outlier
Der verwendete Datensatz beinhaltet keine unplausiblen Ausreißer. Alle in der univariaten Betrachtung auftretenden extremen Werte werden im Datensatz belassen, da sie in der multivariaten Betrachtung nicht auffällig sind

### Clean_data function

In [45]:
def clean_data(target):
    """
    Input: target to be changed to classes 1/0
    Output: cleaned target
    """
    # convert target from g/h --> 1/0
    target = target.map({
    "g": 1,   # gamma = signal
    "h": 0    # hadron = background
    })
  
    
    return target

### set cross-validation-parameter
Im folgenden wird immer wieder auf diese Definition zurückgegriffen

In [46]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


### Resampling

Da die Zielklassen mit etwa 65% (g) zu 35% (h) leicht unausgegleichen sind, wird über eine Kreuzvalidierung mit GridSearch ermittelt, welches Verfahren für diesen Datensatz am geeignetsten für ein Resampling ist

In [47]:
selected_features = features_train.columns   # use all columns

strategies = [   # combination of strategy name, sampling method and class_weights

    # No treatment of class imbalance
    (
        "baseline",
        "passthrough",
        None
    ),

    # Randomly duplicate minority-class observations
    (
        "oversampling",
        RandomOverSampler(random_state=42),
        None
    ),

    # Randomly remove majority-class observations
    (
        "undersampling",
        RandomUnderSampler(random_state=42),
        None
    ),

    # Handle imbalance through Logistic Regression itself
    (
        "class_weights",
        "passthrough",
        "balanced"
    ),

    # Generate synthetic minority observations
    (
        "SMOTE",
        SMOTE(random_state=42),
        None
    )
]


# ------------------------------------------------------------
# Hyperparameter search space
# ------------------------------------------------------------

param_grid = {
    "estimator__C": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}


# ------------------------------------------------------------
# GridSearch for every imbalance strategy
# ------------------------------------------------------------

results = []

best_models = {}

for name, sampler, class_weight in strategies:

    sampling_pipeline = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "sampler",
            sampler
        ),
        (
            "estimator",
            LogisticRegression(
                solver="lbfgs",
                class_weight=class_weight,
                max_iter=1000,
                random_state=42
            )
        )
    ])
    grid_search = GridSearchCV(estimator = sampling_pipeline,
                              param_grid = param_grid,
                               cv = cv,
                              scoring = scoring["tpr_fpr_020"],
                              n_jobs = -1)
    grid_search.fit(features_train, target_train)
    model = grid_search.best_estimator_.named_steps['estimator']
    print(name)
    print(f"{grid_search.scoring} on Validationset: {grid_search.best_score_:.3f}")
    print(model)
    print('#'*11)

   



baseline
make_scorer(tpr_at_fpr, response_method=('decision_function', 'predict_proba'), max_fpr=0.2) on Validationset: 0.724
LogisticRegression(C=0.001, max_iter=1000, random_state=42)
###########
oversampling
make_scorer(tpr_at_fpr, response_method=('decision_function', 'predict_proba'), max_fpr=0.2) on Validationset: 0.737
LogisticRegression(C=1, max_iter=1000, random_state=42)
###########
undersampling
make_scorer(tpr_at_fpr, response_method=('decision_function', 'predict_proba'), max_fpr=0.2) on Validationset: 0.735
LogisticRegression(C=0.01, max_iter=1000, random_state=42)
###########
class_weights
make_scorer(tpr_at_fpr, response_method=('decision_function', 'predict_proba'), max_fpr=0.2) on Validationset: 0.737
LogisticRegression(C=10, class_weight='balanced', max_iter=1000,
                   random_state=42)
###########
SMOTE
make_scorer(tpr_at_fpr, response_method=('decision_function', 'predict_proba'), max_fpr=0.2) on Validationset: 0.735
LogisticRegression(C=0.1, max_iter=

Für alle Verfahren (baseline ausgenommen), zeigt sich ein sehr ähnlicher Score. Der Einfachheit halber wird deshalb class_weights='balanced' verwendet
 

## Baseline model
Als Baseline Modell wird eine einfache logistische Regression mit allen 10 ursprünglichen Features und ohne Sampling verwendet

In [48]:
# select baseline features
baseline_features = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist"
]

In [49]:
# instantiate model
log_reg = LogisticRegression(
    max_iter=10000,
    random_state=42
)

In [50]:
# build pipeline
log_reg_bl = Pipeline(steps = [('scaler', StandardScaler()), 
                                ('model', log_reg)])


In [51]:
# validate baseline model
cv_log_reg = cross_val_score(estimator= log_reg_bl,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"])

print(f"CV best Score: {cv_log_reg.mean():.3f}")


CV best Score: 0.721


In [52]:
# fit pipeline 
log_reg_bl.fit(features_train, target_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](10,)","['fLength','fWidth','fSize',...,'fM3Trans','fAlpha','fDist']"
n_features_in_,int,10
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [53]:
pickle.dump(log_reg_bl, open('log_reg_bl.p', 'wb'))

## Feature Engineering
Wie in der EDA bereits vorgestellt, gibt es einige inteessante neue Features, die aus den alten Features erstellt werden können

In [54]:
# Betrag der symmetrischen Features
features_train["abs_fAsym"] = features_train["fAsym"].abs()
features_train["abs_fM3Long"] = features_train["fM3Long"].abs()
features_train["abs_fM3Trans"] = features_train["fM3Trans"].abs()

In [55]:
# Längen-/Breitenverhältnis
features_train["width_length_ratio"] = features_train["fWidth"] / features_train["fLength"]


In [56]:
# Größe der Hillas-Ellipse
features_train["ellipse_area"] = features_train["fLength"] * features_train["fWidth"] * np.pi


In [57]:
# wie stark dominiert das hellste Pixel die beiden hellsten Pixel
features_train["brightest_pixel_share"] = features_train["fConc1"] / features_train["fConc"]

In [58]:
# Umwandlung von alpha vom Wertebereich 0 bis 90 auf 0 bis 1, 1 entspricht perfekte Ausrichtung zum Kamerazentrum
features_train["alpha_alignment"] = np.cos(np.deg2rad(features_train["fAlpha"]))

In [59]:
features_train.shape

(17014, 17)

In [60]:
def engineer_features(df):
    """
    Input: DataFrame
    Output: Dataframe
    creates new features
    """
    df = df.copy()
    
    # Betrag der symmetrischen Features
    df["abs_fAsym"] = df["fAsym"].abs()
    df["abs_fM3Long"] = df["fM3Long"].abs()
    df["abs_fM3Trans"] = df["fM3Trans"].abs()

    # Längen-/Breitenverhältnis
    df["width_length_ratio"] = df["fWidth"] / df["fLength"]

    # Größe der Hillas-Ellipse
    df["ellipse_area"] = df["fLength"] * df["fWidth"] * np.pi

    # wie stark dominiert das hellste Pixel die beiden hellsten Pixel
    df["brightest_pixel_share"] = df["fConc1"] / df["fConc"]

    # Umwandlung von alpha vom Wertebereich 0 bis 90 auf 0 bis 1, 1 entspricht perfekte Ausrichtung zum Kamerazentrum
    df["alpha_alignment"] = np.cos(np.deg2rad(df["fAlpha"]))

    return(df)

## Scaling
Da das Scaling wird später in die Pipeline integriert und hier deshalb nur instanziiert

In [61]:
scaler = StandardScaler()

## Logistische Regression
Als erstes Modell wird die logistische Regression betrachtet, zunächst ohne PCA, dann mit PCA. Im Anschluss wird mittels Bayesian Optimization eine Optimierung der Hyperparameter vorgenommen

In [62]:
# build unoptimized model

# instantiate model
log_reg = LogisticRegression(
    solver="lbfgs",  
    max_iter=10000,
    random_state=42,
    class_weight='balanced', 
    n_jobs=-1
)


# Instantiating final pipeline with preprocessor and model (logreg)
log_reg_pipe = Pipeline(steps = [('scaler', StandardScaler()), 
                                ('model', log_reg)])

cv_log_reg = cross_val_score(estimator= log_reg_pipe,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)

print(f"CV Score: {cv_log_reg.mean():.3f}")

CV Score: 0.768


zum Vergleich Ergebnis des Baseline Modells: 0.721
Durch Hinzunahme der neu erstellten Features deutliche Verbesserung der Modellgüte



In [64]:
# build unoptimized model with PCA

# instantiate model
log_reg = LogisticRegression(
    solver="lbfgs",  
    max_iter=10000,
    random_state=42,
    class_weight='balanced', 
    n_jobs=-1
)


# Instantiating final pipeline with preprocessor and model (logreg)
log_reg_pipe_PCA = Pipeline(steps = [('scaler', StandardScaler()), 
                                 ('PCA', PCA(n_components=0.95, random_state=42)),
                                ('model', log_reg)])

# validate baseline model
cv_log_reg = cross_val_score(estimator= log_reg_pipe_PCA,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)

print(f"CV Score: {cv_log_reg.mean():.3f}")

CV Score: 0.717


zum Vergleich ohne PCA: 0.768
Durch Hinzunahme der PCA deutliche Verschlechterung der Modellgüte 

In [65]:
# fit pipeline 
log_reg_pipe_PCA.fit(features_train, np.ravel(target_train))

c:\Users\moonp\Documents\Beruflich\Weiterbildung\Stackfuel - Data Science\Portfolio Projekt\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,steps,"[('scaler', ...), ('PCA', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](17,)","['fLength','fWidth','fSize',...,'ellipse_area','brightest_pixel_share', 'alpha_alignment']"
n_features_in_,int,17
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [66]:
pickle.dump(log_reg_pipe_PCA, open('log_reg_PCA.p', 'wb'))

In [67]:
# instantiate model for optimization
log_reg = LogisticRegression(
    solver="liblinear",  
    max_iter=10000,
    random_state=42,
    class_weight='balanced', 
    n_jobs=-1
)

# Instantiating pipeline for optimization
log_reg_pipe_opt = Pipeline(steps = [('scaler', StandardScaler()), 
                                 ('pca', PCA()),
                                ('model', log_reg)])


In [77]:
# tune hyperparameters with Bayesian Optimization

def objective_log_reg(trial):

    
    print(f"\n========== Trial {trial.number} ==========")
    
    # search space
    C = trial.suggest_float("C", 1e-4, 1000, log=True)        # inverse of regularization strength (small --> simple model)
    l1_ratio = trial.suggest_categorical("l1_ratio", [0.0, 1.0])   # 
    n_components = trial.suggest_int("n_components", 1, 17)   # PCA

    
    params_log_reg = {'model__C': C,
            'model__l1_ratio': l1_ratio,
            'pca__n_components': n_components,
            'model__max_iter' : 10000
            }
    
    log_reg_pipe_opt.set_params(**params_log_reg)

    
    # initiating cv
    score_log_reg =  cross_val_score(estimator=log_reg_pipe_opt, 
                             X=features_train, 
                             y=target_train, 
                             scoring=scoring["tpr_fpr_020"],
                             cv=cv,
                             n_jobs=-1).mean()
    
    print(f"CV fertig: {score_log_reg:.4f}")
    
    return score_log_reg

# create a study (aim to maximize score) und setting a seed (random_state) for reproduceability
study_log_reg = optuna.create_study(sampler=TPESampler(seed = 42), direction='maximize')

# perform hyperparamter tuning (while timing the process)
time_start = time.time()
# starting optimization process with our defined function and 50 iterations
study_log_reg.optimize(objective_log_reg, n_trials=50)   # more trial took to long for time limit
time_bayesian = time.time() - time_start

# store result in a data frame 
values_bayesian_log_reg = [50, study_log_reg.best_trial.number, study_log_reg.best_trial.value, time_bayesian]
results_bayesian_log_reg = pd.DataFrame([values_bayesian_log_reg], columns = ['Number of iterations', 
                                                                        'Iteration Number of Optimal Hyperparamters', 
                                                                        'Score', 
                                                                        'Time Elapsed (s)'])
        
          


[I 2026-09-24 16:35:34,961] A new study created in memory with name: no-name-606ace7e-4095-491b-82eb-0c3d3cd8f6d8



========== Trial 0 ==========


[I 2026-09-24 16:35:35,328] Trial 0 finished with value: 0.7188974618058361 and parameters: {'C': 0.041858227295469716, 'l1_ratio': 0.0, 'n_components': 11}. Best is trial 0 with value: 0.7188974618058361.
[I 2026-09-24 16:35:35,536] Trial 1 finished with value: 0.7275275240481771 and parameters: {'C': 0.0012363188277052211, 'l1_ratio': 0.0, 'n_components': 15}. Best is trial 1 with value: 0.7275275240481771.


CV fertig: 0.7189

========== Trial 1 ==========
CV fertig: 0.7275

========== Trial 2 ==========


[I 2026-09-24 16:35:35,790] Trial 2 finished with value: 0.7678010670115593 and parameters: {'C': 1.6136341713591311, 'l1_ratio': 0.0, 'n_components': 17}. Best is trial 2 with value: 0.7678010670115593.
[I 2026-09-24 16:35:35,927] Trial 3 finished with value: 0.701546681755719 and parameters: {'C': 67.1581131106993, 'l1_ratio': 0.0, 'n_components': 4}. Best is trial 2 with value: 0.7678010670115593.


...

CV fertig: 0.7152

========== Trial 46 ==========
CV fertig: 0.7251

========== Trial 47 ==========


[I 2026-09-24 16:35:44,068] Trial 47 finished with value: 0.723301592433918 and parameters: {'C': 8.147896715482602, 'l1_ratio': 1.0, 'n_components': 14}. Best is trial 27 with value: 0.768520491471991.


CV fertig: 0.7233

========== Trial 48 ==========


[I 2026-09-24 16:35:44,276] Trial 48 finished with value: 0.7684306038315414 and parameters: {'C': 81.43147353086842, 'l1_ratio': 0.0, 'n_components': 16}. Best is trial 27 with value: 0.768520491471991.
[I 2026-09-24 16:35:44,473] Trial 49 finished with value: 0.7622271441273947 and parameters: {'C': 1.6613380569504845, 'l1_ratio': 0.0, 'n_components': 15}. Best is trial 27 with value: 0.768520491471991.


CV fertig: 0.7684

========== Trial 49 ==========
CV fertig: 0.7622


In [78]:
# Ausgabe des Ergebnisses
results_bayesian_log_reg

,Number of iterations,Iteration Number of Optimal Hyperparamters,Score,Time Elapsed (s)
0,50,27,0.769,9.511


In [79]:
# Ausgabe der besten Parameter
study_log_reg.best_params

{'C': 859.9521126161015, 'l1_ratio': 0.0, 'n_components': 16}

In [81]:
best_params_log_reg = study_log_reg.best_params
log_reg_pipe_bo = log_reg_pipe_opt.set_params(
            model__C=best_params_log_reg["C"],
            model__l1_ratio=best_params_log_reg["l1_ratio"],
            pca__n_components=best_params_log_reg["n_components"]
)

In [82]:
cv_log_reg_bo = cross_val_score(estimator= log_reg_pipe_bo,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)
#print(cv_log_reg_bo.mean())
print(f"CV Score: {cv_log_reg_bo.mean():.3f}")

CV Score: 0.769


zum Vergleich ohne Optimierung ohne PCA: 0.768

ohne Optimierung mit PCA: 0.717

nur minimale Verbesserung durch Optimierung

In [83]:
log_reg_pipe_bo.fit(features_train, target_train)

c:\Users\moonp\Documents\Beruflich\Weiterbildung\Stackfuel - Data Science\Portfolio Projekt\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,steps,"[('scaler', ...), ('pca', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](17,)","['fLength','fWidth','fSize',...,'ellipse_area','brightest_pixel_share', 'alpha_alignment']"
n_features_in_,int,17
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [84]:
pickle.dump(log_reg_pipe_bo, open('log_reg_opt.p', 'wb'))

## Support Vektor Machine
In EDA mehrere nichtlineare Zusammenhänge entdeckt, deshalb SVM besonders interessant

In [88]:
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="rbf",
        random_state=42
    ))
])

In [89]:
cv_svm = cross_val_score(estimator= svm_pipe,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)

print(f"CV Score: {cv_log_reg.mean():.3f}")

CV Score: 0.717


Zum Vergleich Baseline Modell: 0.721
leichte Verschlechterung im Vergleich zum Baseline Modell

In [90]:
svm_pipe.fit(features_train, target_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](17,)","['fLength','fWidth','fSize',...,'ellipse_area','brightest_pixel_share', 'alpha_alignment']"
n_features_in_,int,17
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [91]:
pickle.dump(svm_pipe, open('svm_simple.p', 'wb'))

In [93]:
svm_pipe_opt = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", SVC(
        kernel="rbf",
        random_state=42
    ))
])

In [96]:
# tune hyperparameters with Bayesian Optimization

def objective_svm(trial):

    
    print(f"\n========== Trial {trial.number} ==========")
    
    # search space
    C = trial.suggest_float("C", 1e-3, 1000, log=True)        # steuert Trade-Off zwischen einfacher Entscheidungsgrenze und möglichst wenig Trainingsfehlern
    gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)   #  bestimmt wie lokal der Einfluss eines einzelnen Trainingspunkts ist
    n_components = trial.suggest_int("n_components", 1, 17)   # PCA  

    
    params_svm = {'model__C': C,
            'model__gamma': gamma,
            'pca__n_components': n_components,
            'model__max_iter' : 10000
            }
    
    svm_pipe_opt.set_params(**params_svm)

    
    # initiating cv
    score_svm =  cross_val_score(estimator=svm_pipe_opt, 
                             X=features_train, 
                             y=target_train, 
                             scoring=scoring["tpr_fpr_020"],
                             cv=cv,
                             n_jobs=-1).mean()
    
    print(f"CV fertig: {score_svm:.4f}")
    
    return score_svm

# create a study (aim to maximize score) und setting a seed (random_state) for reproduceability
study_svm = optuna.create_study(sampler=TPESampler(seed = 42), direction='maximize')

# perform hyperparamter tuning (while timing the process)
time_start = time.time()
# starting optimization process with our defined function and 50 iterations
study_svm.optimize(objective_svm, n_trials=50)   # more trial took to long for time limit
time_bayesian = time.time() - time_start

# store result in a data frame 
values_bayesian_svm = [50, study_svm.best_trial.number, study_svm.best_trial.value, time_bayesian]
results_bayesian_svm = pd.DataFrame([values_bayesian_svm], columns = ['Number of iterations', 
                                                                        'Iteration Number of Optimal Hyperparamters', 
                                                                        'Score', 
                                                                        'Time Elapsed (s)'])
        
          


[I 2026-09-25 08:44:02,041] A new study created in memory with name: no-name-1d5eb619-753d-4dea-8ff3-1a61b0c561af



========== Trial 0 ==========


[I 2026-09-25 08:44:22,308] Trial 0 finished with value: 0.8178714331905261 and parameters: {'C': 0.1767016940294795, 'gamma': 5.669849511478847, 'n_components': 13}. Best is trial 0 with value: 0.8178714331905261.


CV fertig: 0.8179

========== Trial 1 ==========


[I 2026-09-25 08:44:31,233] Trial 1 finished with value: 0.6851851911729043 and parameters: {'C': 3.907967156822881, 'gamma': 0.0006026889128682511, 'n_components': 3}. Best is trial 0 with value: 0.8178714331905261.


...

CV fertig: 0.8961

========== Trial 48 ==========


[I 2026-09-25 08:58:16,043] Trial 48 finished with value: 0.8472670358095546 and parameters: {'C': 31.125442360265122, 'gamma': 0.5547469927348738, 'n_components': 17}. Best is trial 40 with value: 0.9157685312424219.


CV fertig: 0.8473

========== Trial 49 ==========


[I 2026-09-25 08:58:44,396] Trial 49 finished with value: 0.9014756284859754 and parameters: {'C': 0.5930817921680726, 'gamma': 0.03905072959094763, 'n_components': 16}. Best is trial 40 with value: 0.9157685312424219.


CV fertig: 0.9015


In [97]:
# Ausgabe des Ergebnisses
results_bayesian_svm

,Number of iterations,Iteration Number of Optimal Hyperparamters,Score,Time Elapsed (s)
0,50,40,0.916,882.383


In [98]:
# Ausgabe der besten Parameter
study_svm.best_params

{'C': 17.033881600712867, 'gamma': 0.04402725017995011, 'n_components': 16}

In [ ]:
best_params_svm = study_svm.best_params
svm_pipe_bo = svm_pipe_opt.set_params(
            model__C=best_params_svm["C"],
            model__gamma=best_params_svm["gamma"],
            pca__n_components=best_params_svm["n_components"]
)

In [100]:
cv_svm_bo = cross_val_score(estimator= svm_pipe_bo,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)
#print(cv_log_reg_bo.mean())
print(f"CV Score: {cv_svm_bo.mean():.3f}")

CV Score: 0.916


Zum Vergleich ohne Optimierung: 0.717

optimiertes Logistische Regession-Modell: 0.769

Damit ist die optimierte Support Vektor Machine das bisher interessanteste Modell. Die Optimierung hat eine deutliche Verbesserung gegenüber dem einfachen Modell gebracht

In [ ]:
# fit optimized pipeline to trainset
svm_pipe_bo.fit(features_train, target_train)

c:\Users\moonp\Documents\Beruflich\Weiterbildung\Stackfuel - Data Science\Portfolio Projekt\.venv\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


,steps,"[('scaler', ...), ('pca', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](17,)","['fLength','fWidth','fSize',...,'ellipse_area','brightest_pixel_share', 'alpha_alignment']"
n_features_in_,int,17
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [102]:
# save optimized pipeline as pickle
pickle.dump(svm_pipe_bo, open('svm_opt.p', 'wb'))

## Random Forrest

In [105]:
# build unoptimized model

# instantiate model
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)


# Instantiating final pipeline, no need for scaling or PCA in RandomForest
rf_pipe = model_rf
cv_rf = cross_val_score(estimator= rf_pipe,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                           n_jobs=-1)
#print(cv_rf.mean())
print(f"CV Score: {cv_rf.mean():.3f}")

CV Score: 0.917


Zum Vergleich:
Baseline: 0.721
unoptimierte logostische Regression: 0.768
unoptimierte SVM: 0.717

In [ ]:
# fit pipeline to trainset
rf_pipe.fit(features_train, target_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 

In [108]:
# save fitted pipeline as pickle
pickle.dump(rf_pipe, open('rf_simple.p', 'wb'))

In [ ]:
# optimize model with bayesian optimization
def objective_rf(trial):
    """return maximized f1-score"""
   
    # search space
    n_estimators = trial.suggest_int("n_estimators", 50, 250, step=50)
    max_depth = trial.suggest_int("max_depth", 5, 15)  
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10, step=2)   
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 3,10, step=1)   
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    
    params_rf = {"n_estimators": n_estimators,
                'max_depth': max_depth,
              'min_samples_split': min_samples_split,
             'min_samples_leaf': min_samples_leaf,
              'max_features': max_features
            }
    rf_pipe.set_params(**params_rf)

    
    # initiating cv
    score_rf =  cross_val_score(estimator=rf_pipe, 
                             X=features_train, 
                             y=target_train, 
                             scoring=scoring["tpr_fpr_020"],
                             cv=cv,
                             n_jobs=-1).mean()
    
    return score_rf

# create a study (aim to maximize score) und setting a seed (random_state) for reproduceability
study_rf = optuna.create_study(sampler=TPESampler(seed = 42), direction='maximize')

# perform hyperparamter tuning (while timing the process)
time_start = time.time()
# starting optimization process with our defined function and 50 iterations
study_rf.optimize(objective_rf, n_trials=50)
time_bayesian = time.time() - time_start

# store result in a data frame 
values_bayesian_rf = [50, study_rf.best_trial.number, study_rf.best_trial.value, time_bayesian]
results_bayesian_rf = pd.DataFrame([values_bayesian_rf], columns = ['Number of iterations', 
                                                                        'Iteration Number of Optimal Hyperparamters', 
                                                                        'Score', 
                                                                        'Time Elapsed (s)'])

[I 2026-09-25 09:31:50,355] A new study created in memory with name: no-name-7b45eeb8-612d-4be4-a98e-5667b7cd2e18
[I 2026-09-25 09:31:59,330] Trial 0 finished with value: 0.9206230296661548 and parameters: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9206230296661548.
[I 2026-09-25 09:32:04,179] Trial 1 finished with value: 0.9170266348718779 and parameters: {'n_estimators': 50, 'max_depth': 14, 'min_samples_split': 8, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 0 with value: 0.9206230296661548.
[I 2026-09-25 09:32:15,120] Trial 2 finished with value: 0.8833142429876324 and parameters: {'n_estimators': 250, 'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.9206230296661548.
[I 2026-09-25 09:32:21,614] Trial 3 finished with value: 0.8951805432058848 and parameters: {'n_estimators': 150, 'max_depth': 8, 'min_

In [110]:
# Ausgabe des Ergebnisses
results_bayesian_rf

,Number of iterations,Iteration Number of Optimal Hyperparamters,Score,Time Elapsed (s)
0,50,34,0.927,635.701


In [111]:
# Ausgabe der besten Parameter
study_rf.best_params

{'n_estimators': 200,
 'max_depth': 14,
 'min_samples_split': 2,
 'min_samples_leaf': 3,
 'max_features': 'log2'}

In [113]:
best_params_rf = study_rf.best_params
rf_pipe_bo = rf_pipe.set_params(
            n_estimators=best_params_rf["n_estimators"],
            max_depth=best_params_rf["max_depth"],
            min_samples_split=best_params_rf["min_samples_split"],
            min_samples_leaf=best_params_rf["min_samples_leaf"],
            max_features=best_params_rf["max_features"]
)


In [114]:
cv_rf_bo = cross_val_score(estimator= rf_pipe_bo,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)
#print(cv_log_reg_bo.mean())
print(f"CV Score: {cv_rf_bo.mean():.3f}")

CV Score: 0.927


Zum Vergleich: 

RandomForest ohne Optimierung: 0.917

optimierte logistische Regression: 0.769

optminierte SVM: 0.916

leichte Verbesserung gegenüber unoptimiertem Modell und optimierter SVM, bisher bestes Modell

## Einfaches neuronales Netz 
Für eher kleineren Datensatz wird zunächst einfaches neuronales Netz aus sklearn verwendet, um zu prüfen, ob ein neuronales Netz für dieses Projekt vielversprechend ist

In [120]:
# build unoptimized model

# instantiate model
model_nn = MLPClassifier(
        hidden_layer_sizes=(32, 16),   # zwei hidden layer, 1. mit  32 dann mit 16 
        activation="relu",             # acivation function
        solver="adam",
        alpha=1e-4,                    # L2 Regularisierung
        learning_rate_init=1e-3,       # anfängliche Schrittweite
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=42
    )



nn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", model_nn)
])

cv_nn = cross_val_score(estimator= nn_pipe,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                           n_jobs=-1)

print(f"CV Score: {cv_nn.mean():.3f}")

CV Score: 0.904


Zum Vergleich: 

Baseline: 0.721

unoptimierte logistiche Regression: 0.768

unoptimierte SVM: 0.717

unoptimierter RandomForest: 0.917

In [121]:
# fit pipeline to trainset
nn_pipe.fit(features_train, target_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](17,)","['fLength','fWidth','fSize',...,'ellipse_area','brightest_pixel_share', 'alpha_alignment']"
n_features_in_,int,17
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [122]:
# save fitted pipeline as pickle
pickle.dump(nn_pipe, open('nn_simple.p', 'wb'))

In [136]:
# optimize model with bayesian optimization
architectures = {
        "16": (16,),
        "32": (32,),
        "64": (64,),
        "32_16": (32, 16),
        "64_32": (64, 32),
        "64_32_16": (64, 32, 16)
}


def objective_nn(trial):
    """return maximized score"""


    architecture_name = trial.suggest_categorical(
        "architecture",
        list(architectures.keys())
    )
    
   
    # search space
    hidden_layer_sizes = architectures[architecture_name]
    alpha = trial.suggest_float("alpha", 1e-6, 1e-2, log=True)  
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)   
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])  

    
    params_nn = {"model__hidden_layer_sizes": hidden_layer_sizes,
                'model__alpha': alpha,
              'model__learning_rate_init': learning_rate_init,
             'model__activation': activation  
            }
    nn_pipe.set_params(**params_nn)

    
    # initiating cv
    score_nn =  cross_val_score(estimator=nn_pipe, 
                             X=features_train, 
                             y=target_train, 
                             scoring=scoring["tpr_fpr_020"],
                             cv=cv,
                             n_jobs=-1).mean()
    
    return score_nn

# create a study (aim to maximize score) und setting a seed (random_state) for reproduceability
study_nn = optuna.create_study(sampler=TPESampler(seed = 42), direction='maximize')

# perform hyperparamter tuning (while timing the process)
time_start = time.time()
# starting optimization process with our defined function and 50 iterations
study_nn.optimize(objective_nn, n_trials=50)
time_bayesian = time.time() - time_start

# store result in a data frame 
values_bayesian_nn = [50, study_nn.best_trial.number, study_nn.best_trial.value, time_bayesian]
results_bayesian_nn = pd.DataFrame([values_bayesian_nn], columns = ['Number of iterations', 
                                                                        'Iteration Number of Optimal Hyperparamters', 
                                                                        'Score', 
                                                                        'Time Elapsed (s)'])

[I 2026-09-25 11:19:40,998] A new study created in memory with name: no-name-65c78519-ee7e-4cbc-b424-313d634902e7


[I 2026-09-25 11:19:50,388] Trial 0 finished with value: 0.8978794357772208 and parameters: {'architecture': '32', 'alpha': 1.7073967431528114e-06, 'learning_rate_init': 0.005399484409787433, 'activation': 'tanh'}. Best is trial 0 with value: 0.8978794357772208.
[I 2026-09-25 11:19:59,699] Trial 1 finished with value: 0.903902190607065 and parameters: {'architecture': '32', 'alpha': 1.6480446427978994e-05, 'learning_rate_init': 0.0011207606211860567, 'activation': 'relu'}. Best is trial 1 with value: 0.903902190607065.
[I 2026-09-25 11:20:04,730] Trial 2 finished with value: 0.9081283243068465 and parameters: {'architecture': '64_32_16', 'alpha': 6.290644294586145e-06, 'learning_rate_init': 0.0010677482709481358, 'activation': 'relu'}. Best is trial 2 with value: 0.9081283243068465.
[I 2026-09-25 11:20:12,253] Trial 3 finished with value: 0.9023739794681109 and parameters: {'architecture': '64_32', 'alpha': 1.6536937182824424e-05, 'learning_rate_init': 0.0001567993391672301, 'activatio

In [140]:
# Ausgabe des Ergebnisses
results_bayesian_nn

,Number of iterations,Iteration Number of Optimal Hyperparamters,Score,Time Elapsed (s)
0,50,36,0.913,349.652


In [141]:
# Ausgabe der besten Parameter
study_nn.best_params

{'architecture': '32',
 'alpha': 4.314125869757452e-05,
 'learning_rate_init': 0.006015050910243471,
 'activation': 'relu'}

In [143]:
best_params_nn = study_nn.best_params

best_hidden_layer_sizes = architectures[
    best_params_nn["architecture"]
]

nn_pipe_bo = nn_pipe.set_params(
            model__hidden_layer_sizes=best_hidden_layer_sizes,
            model__alpha=best_params_nn["alpha"],
            model__learning_rate_init=best_params_nn["learning_rate_init"],
            model__activation=best_params_nn["activation"]
)


In [144]:
cv_nn_bo = cross_val_score(estimator= nn_pipe_bo,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)
#print(cv_log_reg_bo.mean())
print(f"CV Score: {cv_nn_bo.mean():.3f}")

CV Score: 0.913


Zum Vergleich: 

neuronales Netz ohne Optimierung: 0.904

optimierte logistische Regression: 0.769

optminierte SVM: 0.916

optimierter RandomForest: 0.927

optimierter RandomForest bisher bestes Modell

In [146]:
# optimize model with bayesian optimization
def objective_rf_2(trial):
    """return maximized f1-score"""
   
    # search space
    n_estimators = trial.suggest_int("n_estimators", 300, 1500, step=100)
    max_depth = trial.suggest_int("max_depth", 4, 30)  
    min_samples_split = trial.suggest_int("min_samples_split", 2, 30)   
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1,20, step=1)   
    max_features = trial.suggest_float("max_features", 0.3, 1.0)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"])
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"])
    max_samples = trial.suggest_float("max_samples", 0.6, 1.0)
    

    
    params_rf_2 = {"n_estimators": n_estimators,
                'max_depth': max_depth,
              'min_samples_split': min_samples_split,
             'min_samples_leaf': min_samples_leaf,
              'max_features': max_features,
              'class_weight': class_weight,
              'criterion': criterion,
              'max_samples': max_samples
            }
    rf_pipe.set_params(**params_rf_2)

    
    # initiating cv
    score_rf_2 =  cross_val_score(estimator=rf_pipe, 
                             X=features_train, 
                             y=target_train, 
                             scoring=scoring["tpr_fpr_020"],
                             cv=cv,
                             n_jobs=-1).mean()
    
    return score_rf_2

# create a study (aim to maximize score) und setting a seed (random_state) for reproduceability
study_rf_2 = optuna.create_study(sampler=TPESampler(seed = 42), direction='maximize')

# perform hyperparamter tuning (while timing the process)
time_start = time.time()
# starting optimization process with our defined function and 50 iterations
study_rf_2.optimize(objective_rf_2, n_trials=50)
time_bayesian = time.time() - time_start

# store result in a data frame 
values_bayesian_rf_2 = [50, study_rf_2.best_trial.number, study_rf_2.best_trial.value, time_bayesian]
results_bayesian_rf_2 = pd.DataFrame([values_bayesian_rf_2], columns = ['Number of iterations', 
                                                                        'Iteration Number of Optimal Hyperparamters', 
                                                                        'Score', 
                                                                        'Time Elapsed (s)'])

[I 2026-09-25 13:26:33,421] A new study created in memory with name: no-name-266634af-56b2-435c-8fdf-2b06251099f3
[I 2026-09-25 13:36:32,132] Trial 0 finished with value: 0.9200840271602942 and parameters: {'n_estimators': 700, 'max_depth': 29, 'min_samples_split': 23, 'min_samples_leaf': 12, 'max_features': 0.40921304830970556, 'class_weight': 'balanced_subsample', 'criterion': 'entropy', 'max_samples': 0.9879639408647978}. Best is trial 0 with value: 0.9200840271602942.
[I 2026-09-25 13:43:00,852] Trial 1 finished with value: 0.9086657909627356 and parameters: {'n_estimators': 1300, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 0.5129695700716763, 'class_weight': None, 'criterion': 'gini', 'max_samples': 0.7465447373174767}. Best is trial 0 with value: 0.9200840271602942.
[I 2026-09-25 13:53:07,240] Trial 2 finished with value: 0.9204433756365693 and parameters: {'n_estimators': 800, 'max_depth': 25, 'min_samples_split': 7, 'min_samples_leaf': 11, 'ma

In [147]:
# Ausgabe des Ergebnisses
results_bayesian_rf_2

,Number of iterations,Iteration Number of Optimal Hyperparamters,Score,Time Elapsed (s)
0,50,49,0.931,"9,694.861"


In [148]:
# Ausgabe der besten Parameter
study_rf_2.best_params

{'n_estimators': 800,
 'max_depth': 28,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 0.3838170996854309,
 'class_weight': None,
 'criterion': 'log_loss',
 'max_samples': 0.9553039046686342}

In [150]:
best_params_rf_2 = study_rf_2.best_params
rf_pipe_bo_2 = rf_pipe.set_params(
            n_estimators=best_params_rf_2["n_estimators"],
            max_depth=best_params_rf_2["max_depth"],
            min_samples_split=best_params_rf_2["min_samples_split"],
            min_samples_leaf=best_params_rf_2["min_samples_leaf"],
            max_features=best_params_rf_2["max_features"],
            class_weight=best_params_rf_2["class_weight"],
            criterion=best_params_rf_2["criterion"],
            max_samples=best_params_rf_2["max_samples"]
)

In [151]:
cv_rf_bo_2 = cross_val_score(estimator= rf_pipe_bo_2,
                            X=features_train,
                            y=target_train,
                            cv=cv,
                            scoring=scoring["tpr_fpr_020"],
                            n_jobs=-1)
#print(cv_log_reg_bo.mean())
print(f"CV Score: {cv_rf_bo_2.mean():.3f}")

CV Score: 0.931


Zum Vergleich: 

optimierte logistische Regression: 0.769

optminierte SVM: 0.916

optimierter RandomForest: 0.927

optimiertes neuronales Netz: 0.913